# Figure 2 — OHLC selections of an excursion

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from fast_excursion_limit import plot_config
from fast_excursion_limit.fast_excursion_heston import FastExcursionHeston
from fast_excursion_limit.selections import constant_runs, excursion_size

plot_config.set_style()

SAVE_PLOT = True

## Parameters

In [ ]:
from fast_excursion_limit import defaults

seed = defaults.SEED

maturity = defaults.MATURITY_PATHWISE
step_size = defaults.DELTA_SCALE_PATHWISE * maturity

zoom_pad_time = 0.01

## Simulate and find the largest excursion

In [ ]:
model = FastExcursionHeston.example()

np.random.seed(seed)
Z, Y = model.simulate(maturity=maturity, step_size=step_size)

# Pick the run with the biggest excursion size
start, end = max(constant_runs(Y), key=lambda run: excursion_size(Z, run))

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=plot_config.TWO_PANEL_FIGSIZE, sharey=True)
for ax in axes:
    ax.set_box_aspect(1)

# left: full path
axes[0].plot(Y, Z)
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Price")

# right: zoom window around the excursion
lo = np.searchsorted(Y, Y[start] - zoom_pad_time, side="left")
hi = np.searchsorted(Y, Y[end] + zoom_pad_time, side="right") - 1
axes[0].axvspan(Y[lo], Y[hi], color="gray", alpha=0.2)

t = Y[start]
open_price, close_price = Z[start], Z[end]
high, low = Z[start : end + 1].max(), Z[start : end + 1].min()

axes[1].plot(Y[lo : hi + 1], Z[lo : hi + 1])
axes[1].plot(t, open_price, "s", fillstyle="none", label="$O_t$", color="C1")
axes[1].plot(t, high, "^", fillstyle="none", label="$H_t$", color="C1")
axes[1].plot(t, low, "x", fillstyle="none", label="$L_t$", color="C1")
axes[1].plot(t, close_price, "o", fillstyle="none", label="$C_t$", color="C1")
axes[1].legend()
axes[1].set_xlabel("Time")

fig.tight_layout()
if SAVE_PLOT:
    fig.savefig(plot_config.PLOTS_DIR / "figure-2.pdf")